# Equity compensation tax planning with Claude and the OptionsAhoy MCP server

[OptionsAhoy](https://optionsahoy.com) (AlphaLatitude Inc.) runs a remote Model Context Protocol (MCP) server at `https://optionsahoy.com/mcp` that exposes seven deterministic US equity-compensation tax tools:

- `amt_iso_optimize`, `nso_calculate`, `rsu_sell_vs_hold`, `concentration_analyze`, `protective_put_price`, `qsbs_check`, `equity_funding_plan`

The server uses streamable HTTP transport, is free, and requires no authentication or account. Every tool is annotated read-only, computes against the full US federal tax code plus all 50 states and DC, and returns MCP `structuredContent` matching a declared `outputSchema`. The same inputs always produce the same outputs.

In this recipe, Claude connects to the server through the Claude API's MCP connector and answers a multi-year incentive stock option (ISO) exercise question by calling `amt_iso_optimize` instead of doing the tax arithmetic in context.

## Why route the math to a calculator

Multi-year exercise planning is a search problem, not a recall problem. The answer depends on the alternative minimum tax (AMT) crossover point in each year, AMT credit recovery in later years, federal and state bracket interactions, and the choice of how many shares to exercise in each of the plan's years. The feasible schedule space is far too large to evaluate token by token.

In a published benchmark, five LLMs given this same multi-year ISO problem overshot the achievable after-tax outcome by 2x to 20x when they did the arithmetic in context ([writeup](https://optionsahoy.com/why-deterministic)). With the MCP connector, Claude does what it is good at (parsing the question, choosing the tool, explaining the result) and the server does the part that has exactly one right answer.

**Scope and limitations**: OptionsAhoy is a planning calculator, not tax advice; confirm decisions with a tax professional. Coverage is US-only (federal, all 50 states, and DC).

Documentation for agents: https://optionsahoy.com/for-agents. Full tool and schema reference: https://optionsahoy.com/llms-full.txt

## Step 1: Set up the environment

Install the Anthropic SDK (`requests` is used later for the REST alternative) and make sure your Claude API key is available as the `ANTHROPIC_API_KEY` environment variable.

In [ ]:
%pip install anthropic requests

In [ ]:
import os

from anthropic import Anthropic

if "ANTHROPIC_API_KEY" not in os.environ:
    raise RuntimeError(
        "Set the ANTHROPIC_API_KEY environment variable before running this notebook."
    )

client = Anthropic()
MODEL_NAME = "claude-opus-4-8"

## Step 2: Connect Claude to the MCP server and ask the question

The MCP connector is a beta feature of the Messages API (beta flag `mcp-client-2025-11-20`). You pass the server's URL in the `mcp_servers` parameter and the API handles the connection server-side: no local MCP client, no subprocess, no tool schemas to copy into your request. Claude discovers the seven tools from the server and decides when to call them.

The question below deliberately contains every input `amt_iso_optimize` requires: share count, strike, current fair market value (FMV), income, filing status, state, employment status, and grant date. The ticker lets the server resolve growth and volatility assumptions from cached market data. If a required field were missing, the tool would return a validation error and Claude would ask you for it rather than invent a value.

In [ ]:
USER_PROMPT = (
    "I have 8,000 ISOs at a $5 strike, FMV $40, income $200K, single filer in California, "
    "still employed, granted 2024-01-15. How many should I exercise this year to stay under "
    "the AMT crossover, and what does a 5-year plan look like?"
)

response = client.beta.messages.create(
    model=MODEL_NAME,
    max_tokens=16000,
    betas=["mcp-client-2025-11-20"],
    mcp_servers=[
        {
            "type": "url",
            "url": "https://optionsahoy.com/mcp",
            "name": "optionsahoy",
        }
    ],
    messages=[{"role": "user", "content": USER_PROMPT}],
)

print(f"stop_reason: {response.stop_reason}")

## Step 3: Inspect the tool calls and the final answer

The response content interleaves three block types:

- `mcp_tool_use`: the tool Claude decided to call, with the exact input it constructed from your question
- `mcp_tool_result`: what the server returned (serialized JSON text plus structured content)
- `text`: Claude's own explanation

In [ ]:
import json

for block in response.content:
    if block.type == "mcp_tool_use":
        print(f"=== mcp_tool_use: {block.name} (server: {block.server_name}) ===")
        print(json.dumps(block.input, indent=2))
        print()
    elif block.type == "mcp_tool_result":
        label = "error" if block.is_error else "ok"
        print(f"=== mcp_tool_result ({label}) ===")
        for item in block.content:
            if item.type == "text":
                preview = item.text[:600]
                suffix = " ..." if len(item.text) > 600 else ""
                print(preview + suffix)
        print()
    elif block.type == "text":
        print(block.text)

You should see Claude call `amt_iso_optimize` once, with the inputs parsed from the question. The result contains, among other fields:

- `crossoverShares`: the maximum shares exercisable in year 1 before tentative AMT exceeds regular tax (1,321 shares for this example)
- `schedules`: three full plans (`lumpSum`, `evenSplit`, `optimized`), each with the after-tax Net Final Value (NFV) at the 5-year horizon

Claude's final text block turns that into a per-year exercise plan with the dollar consequences of each alternative.

## The other six tools

The same connection gives Claude six more deterministic calculators. Each follows the same pattern: required fields must come from the user (or resolve from an optional ticker), and the result is structured JSON with a declared schema.

| Tool | What it computes |
|---|---|
| `nso_calculate` | After-tax payout on a non-qualified stock option (NSO) exercise: federal, state, and FICA, comparing sell-at-exercise vs hold-for-long-term-capital-gains |
| `rsu_sell_vs_hold` | After-tax payout on a restricted stock unit (RSU) vest, including the gap between 22% supplemental withholding and your marginal bracket, and sell-at-vest vs hold |
| `concentration_analyze` | Single-stock concentration risk on an existing position, comparing sell-down, hold, and optionally a hedged hold |
| `protective_put_price` | Black-Scholes pricing of a protective put or zero-cost collar on a single-stock position |
| `qsbs_check` | Section 1202 qualified small business stock (QSBS) qualification against the eight statutory tests, with per-state conformity |
| `equity_funding_plan` | Multi-year sale schedule to fund a target after-tax amount by a deadline, returning four named plans on the risk/wealth frontier |

The server also publishes seven MCP resources (topical briefings) and seven MCP prompts that scaffold typical questions onto the right tool.

## REST alternative

Every tool is also exposed as plain REST at `POST https://optionsahoy.com/api/v1/<tool-slug>` (same engine, same results), with an OpenAPI 3.1 spec at https://optionsahoy.com/openapi.json. This is useful when you want the numbers without a model in the loop, or when you build the tool loop yourself. The response wraps the tool result in `{"ok": true, "result": {...}}`.

In [ ]:
import requests

payload = {
    "shares": 8000,
    "strike": 5,
    "fmv": 40,
    "horizon": 5,
    "ticker": "NVDA",
    "expectedSalePrice": 80,
    "ordinaryIncome": 200000,
    "filingStatus": "single",
    "stateCode": "CA",
    "stillEmployed": True,
    "hasLeftCompany": False,
    "grantDate": "2024-01-15",
    "carryforwardCredit": 0,
    "cashReturnRate": 0.05,
}

rest_response = requests.post("https://optionsahoy.com/api/v1/amt-iso", json=payload, timeout=30)
rest_response.raise_for_status()
result = rest_response.json()["result"]

print(f"AMT crossover shares (year 1): {result['crossoverShares']:,}")
for name, schedule in result["schedules"].items():
    print(f"{name:>9} NFV: ${schedule['nfv']:,.0f}")

For local development there is also a stdio transport: `npx -y optionsahoy-mcp` (published to npm with provenance).

## Wrapping up

You connected Claude to a remote MCP server with three lines of request configuration, and the model handed a search problem to a deterministic engine instead of approximating it. The same pattern applies to any read-only computational MCP server: let the model own the language and the routing, and let the tool own the math.

- Agent documentation: https://optionsahoy.com/for-agents
- Full schema reference: https://optionsahoy.com/llms-full.txt
- Why deterministic tax math: https://optionsahoy.com/why-deterministic